In [ ]:

import os
from dotenv import load_dotenv
load_dotenv()

from typing import Annotated
from typing_extensions import TypedDict

from langchain.chat_models import init_chat_model
from langchain.messages import HumanMessage, SystemMessage
from langchain.tools import tool

from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.types import interrupt, Command
from langgraph.checkpoint.memory import InMemorySaver


# Model
llm = init_chat_model("groq:openai/gpt-oss-20b")


# Tool
@tool
def Sendemail(to: str, subject: str, body: str) -> str:
    """Send an email after getting human approval."""

    approval = interrupt({
        "question": "Do you approve sending this email?",
        "to": to,
        "subject": subject,
        "body": body
    })

    if approval["decision"] == "yes":
        # Actual email sending logic goes here
        return "Email successfully submitted."

    return "Email rejected by user. Do not send."


# Bind tool
tools = [Sendemail]
llmWithTools = llm.bind_tools(tools)


# State
class State(TypedDict):
    messages: Annotated[list, add_messages]


# Chatbot node
def chatbot(state: State):

    response = llmWithTools.invoke([
        SystemMessage(content="""
        You are an email assistant.

        1. Collect the recipient, subject and email body.
        2. Prepare the email.
        3. When ready, call Sendemail.
        4. After the tool returns successfully, tell the user
           that the email was submitted.
        5. If the tool reports rejection, tell the user it was rejected.
        6. Do not call the tool again after it has completed.
        """),
        *state["messages"]
    ])

    return {"messages": [response]}


# Graph
checkpointer = InMemorySaver()

graph_builder = StateGraph(State)

graph_builder.add_node("chatbot", chatbot)
graph_builder.add_node("tools", ToolNode(tools))

graph_builder.add_edge(START, "chatbot")

graph_builder.add_conditional_edges(
    "chatbot",
    tools_condition,
    {
        "tools": "tools",
        END: END
    }
)

graph_builder.add_edge("tools", "chatbot")

graph = graph_builder.compile(checkpointer=checkpointer)


# Run
config = {
    "configurable": {
        "thread_id": "1"
    }
}

for chunk in graph.stream(
    {
        "messages": [
            HumanMessage(content="Prepare an email to hr@example.com requesting leave from 10 to 12 November.")
        ]
    },
    config=config,
    stream_mode="values"
):
    if chunk.get("messages"):
        last_message = chunk["messages"][-1]

        if hasattr(last_message, "content") and last_message.content:
            print(last_message.content)

    if chunk.get("__interrupt__"):
        print("\nApproval required:")
        print(chunk["__interrupt__"])


# Resume after user approval
result = graph.invoke(
    Command(resume={"decision": "yes"}),
    config=config
)

print("\nFinal response:")
print(result["messages"][-1].content)


Human approval required: (Interrupt(value='Do you approve to send email?', id='edf607c5988f3a7beef64df521928027', response_schema=None),)
**Subject:** Urgent Leave Request – Family Emergency  

Dear HR Manager,

I hope this message finds you well. I am writing to request urgent leave due to a family emergency. I would like to take leave from **10 November 2026** to **12 November 2026**.

To ensure a smooth workflow during my absence, I will complete all pending tasks and hand over any critical responsibilities before my departure. I will also remain reachable for urgent queries if needed.

Thank you for your understanding and prompt consideration.

Kind regards,  
[Your Name]  
[Your Position]  
[Your Contact Information]

---

Please review the draft above. Let me know if you approve or if any adjustments are needed before I send it.
